# Aula 01 — Setup completo: do zero à primeira chamada LLM

**Objetivo:** sair com ambiente funcionando (`uv` + `.env` + Groq) e entender cada peça: `Human/AIMessage`, `invoke vs stream`, `temperature`, `getpass`.

**Como usar:** rode em ordem. Este notebook é autocontido (funciona até no Colab com 1 upload). Sem `GROQ_API_KEY`, as células usam Fake offline (avisam). Com a chave, usam Groq real.
Espelho executável local: `python src/aula01_setup.py`.

In [ ]:
# CÉLULA 0 — instalar dependências
# No Colab estas linhas são necessárias (o ambiente vem cru).
# Localmente você já rodou `uv sync`; o comando abaixo é inofensivo (no-op) se tudo existe.
%pip install -q langgraph langchain-groq python-dotenv
print('dependências prontas.')

## 1. `uv` vs `pip install` — por que `uv`?

Você conhece o fluxo clássico: `python -m venv .venv → source .venv/bin/activate → pip install -r requirements.txt`.
O `uv` (mesmo ecossistema do curso, que usa `uv.lock`/`pyproject.toml`) faz o mesmo trabalho, com 3 diferenças práticas:

1. **Velocidade:** o `uv` é escrito em Rust e resolve+baixa pacotes de 10x a 100x mais rápido que o `pip`. Num projeto com `torch`/`transformers` isso é minutos vs dezenas de minutos.
2. **Reprodutibilidade (`uv.lock`):** `pip install -r requirements.txt` fixa só o que você listou (e `langgraph` solto puxa “a última versão de hoje” — amanhã pode ser outra). O `uv lock/sync` trava **toda a árvore** de dependências no `uv.lock`; outra máquina com `uv sync` instala byte a byte igual.
3. **Tudo numa ferramenta só:** venv + installer + lock + runner. Comparativo:

| Tarefa | pip clássico | uv |
|---|---|---|
| criar venv | `python -m venv .venv` | `uv venv` |
| instalar tudo travado | `pip install -r requirements.txt` | `uv sync` (lê `pyproject.toml` + `uv.lock`) |
| adicionar pacote | `pip install X` + editar requirements na mão | `uv add X` (atualiza `pyproject` + lock sozinho) |
| rodar sem ativar venv | (precisa `source .../activate`) | `uv run python src/aula01_setup.py` |
| rodar testes | `pytest` (com venv ativo) | `uv run pytest` |

Quando `pip` ainda vale: dentro do Colab (`%pip install`, sem terminal/venv próprio) e em tutoriais antigos como o `module-0` do curso — lá use `pip`, aqui usamos `uv`.
Para este projeto, o essencial é só: `uv venv && uv sync` (ou `uv run ...` direto, que cria o venv sozinho se faltar).

In [ ]:
# CÉLULA 1 — raio-X do ambiente (100% offline)
import sys
print('python:', sys.version.split()[0])
for pkg in ('langgraph', 'langchain_core', 'langchain_groq', 'dotenv'):
    try:
        mod = __import__(pkg)
        print(f'{pkg}:', getattr(mod, '__version__', 'instalado'))
    except ImportError:
        print(f'{pkg}: FALTANDO -> local: uv sync | colab: %pip install -q langgraph langchain-groq python-dotenv')

## 2. Chave sem vazar: `.env` + `getpass`

Regra de ouro (vale o curso inteiro): **chave mora fora do código**. O padrão deste repo:

- `.env.example` versionado (só nomes, ex: `GROQ_API_KEY=` vazio) → cada máquina copia para `.env` e preenche.
- `.env` real **nunca** entra no git (está no `.gitignore`).
- `load_dotenv()` lê o `.env`; no Colab (sem arquivo), `getpass.getpass()` pede digitação oculta.

A célula abaixo tenta nesta ordem: variável de ambiente → `.env` → `getpass` interativo → modo Fake (se nada funcionar, ex: execução automática).

In [ ]:
# CÉLULA 2 — obter a chave sem expô-la
import getpass
import os
from dotenv import load_dotenv

load_dotenv()  # lê .env se existir; fora isso, nada muda

if not os.getenv('GROQ_API_KEY'):
    try:
        os.environ['GROQ_API_KEY'] = getpass.getpass('GROQ_API_KEY (digitação oculta): ')
    except Exception:
        print('sem entrada interativa -> seguirei em modo Fake offline.')

print('chave presente:', bool(os.getenv('GROQ_API_KEY')))
# NUNCA dê print na chave. Verificamos só presença (True/False).

## 3. `ChatGroq` e `temperature` — o que são de verdade

- `ChatGroq(model='openai/gpt-oss-20b', temperature=0)` é um **cliente de API**: recebe lista de mensagens, devolve `AIMessage`. O curso usa `ChatOpenAI(model='gpt-4o-mini')` — a interface é idêntica, só muda o fornecedor (aqui Groq, por latência e custo).
- `temperature` não é matemática para você decorar fórmula: é o botão **determinismo vs variedade**. `0` = mesma pergunta, (quase) sempre mesma resposta — use nos estudos e em agentes que chamam tools. `0.9` = respostas variadas — use em texto criativo. Nos exercícios de grafo, sempre `0`.
- `HumanMessage(content=...)` / `AIMessage(content=...)` são **structs**: quem falou + o texto (+ metadados). Quando você passa string pura no `invoke`, ela é convertida para `HumanMessage` automaticamente.

In [ ]:
# CÉLULA 3 — primeira chamada (Groq real se houver chave, Fake se não)
import os

MODELO = 'openai/gpt-oss-20b'

def get_llm(temperature=0, reasoning_effort='low'):
    """Groq (com reasoning_effort) ou Fake offline.

    reasoning_effort controla o ORÇAMENTO de tokens gastos raciocinando
    antes de responder (low/medium/high; não existe 'off'). Sem ele, o
    gpt-oss gasta tudo pensando e devolve content vazio (veja seção 3.1).
    """
    if os.getenv('GROQ_API_KEY'):
        try:
            from langchain_groq import ChatGroq
        except ImportError:
            print('AVISO: langchain_groq ausente -> rode %pip install -q langchain-groq')
            print('       seguirei no Fake offline até instalar.')
        else:
            return ChatGroq(model=MODELO, temperature=temperature,
                             reasoning_effort=reasoning_effort), 'groq'
    from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
    from langchain_core.messages import AIMessage
    print('AVISO: modo Fake offline (mecânica igual, texto simulado).')
    fake = FakeMessagesListChatModel(
        responses=[AIMessage(content='Olá! Sou uma resposta simulada para estudo offline.')]
    )
    return fake, 'fake'

llm, origem = get_llm(temperature=0)
print(f'origem={origem} modelo={MODELO}')
resposta = llm.invoke('Como vai você? Responda em uma frase.')
print('tipo:', type(resposta).__name__)
print('conteúdo:', resposta.content)

## 3.1 Por que o `content` veio VAZIO? (modelos de raciocínio)

Sintoma: sem erro nenhum, `resposta.content` sai `''` e o texto aparece em
`resposta.additional_kwargs['reasoning_content']`. Diagnóstico no `token_usage`:

```
completion_tokens_details: {'reasoning_tokens': 2046}
finish_reason: 'length'   <- acabou o token ANTES de escrever a resposta
```

**O que está acontecendo:** `openai/gpt-oss-20b` é um modelo de *raciocínio*.
Ele gasta tokens pensando em `additional_kwargs['reasoning_content']` e **só
depois** escreve a resposta final em `content`. Se o raciocínio consome todo o
limite de tokens de conclusão (2048 no Groq), a resposta nunca começa — daí o
`content` vazio. Não é bug seu e não é bug da API: é o orçamento estourando
na fase de análise.

**`reasoning_effort`** (parâmetro do Groq) controla esse orçamento:

| Valor | Efeito | Quando usar |
|---|---|---|
| `low` | pouco raciocínio (medido: 10 tokens), responde na hora | estudos, **agentes com tools**, queries objetivas |
| `medium` | equilibrado | análise moderada |
| `high` | raciocínio longo | problema difícil, prova de código |

Não existe `none`/desligar — a API rejeita com 400 (`must be one of low, medium, high`).
Analogia: `timeout` de um worker que analisa e *depois* produz; se a análise
leva todo o tempo, a produção não acontece e você recebe só o log da análise.

**Por que isso é crítico para as próximas aulas:** com raciocínio longo, o
`tool_calls` vem vazio — o LLM nunca pede a ferramenta e o agente não age.

In [ ]:
# CÉLULA 3.1 — diagnóstico: separar RACIOCÍNIO de RESPOSTA
# Rode esta célula se o `content` vier vazio. O extrator trata os dois casos.
import os

def get_llm_diag(temperature=0, reasoning_effort='low'):
    if os.getenv('GROQ_API_KEY'):
        try:
            from langchain_groq import ChatGroq
        except ImportError:
            print('AVISO: langchain_groq ausente -> rode %pip install -q langchain-groq')
            print('       seguirei no Fake offline até instalar.')
        else:
            return ChatGroq(model=MODELO, temperature=temperature,
                            reasoning_effort=reasoning_effort), 'groq'
    print('AVISO: modo Fake offline (mecânica igual, texto simulado).')
    fake = FakeMessagesListChatModel(
        responses=[AIMessage(content='resposta simulada sem raciocínio.')]
    )
    return fake, 'fake'

def texto_final(msg) -> str:
    if msg.content:
        return msg.content
    raciocinio = msg.additional_kwargs.get('reasoning_content')
    if raciocinio:
        print('AVISO: content vazio; o modelo gastou o orçamento raciocinando.')
        print('       correção: reasoning_effort="low"')
        return '[raciocínio] ' + raciocinio[:200] + '...'
    return ''

llm2, origem2 = get_llm_diag(temperature=0, reasoning_effort='low')
r2 = llm2.invoke('Como vai você? Responda em uma frase.')
print('origem:', origem2)
print('content:', repr(r2.content))
print('raciocínio (se houve):', repr(r2.additional_kwargs.get('reasoning_content', '')[:80]))
print('tokens:', r2.response_metadata.get('token_usage', {}).get('completion_tokens_details'))
print('finish_reason:', r2.response_metadata.get('finish_reason'))
print('texto_final ->', repr(texto_final(r2)[:90]))


## 4. Anatomia do `AIMessage` (abra o capô)

- `content: str` — o texto. É o que você exibe.
- `tool_calls: list` — vazio por enquanto (`[]`). Na Aula 03 ele virá preenchido quando o LLM pedir ferramenta — é o campo que o roteador lê.
- `response_metadata` — bastidor da API: modelo usado, tempos, e (no Groq real) `token_usage` = quantos tokens você gastou. No Fake, mínimo.
- `id` — identificador único da mensagem (na Aula 08 ele permite sobrescrever/apagar mensagens).

In [ ]:
# CÉLULA 4 — inspecionar a resposta (reusa o llm da célula anterior)
r = llm.invoke('Diga apenas: ok')
print('content:', repr(r.content))
print('tool_calls:', r.tool_calls)
print('metadata:', r.response_metadata)
print('id:', getattr(r, 'id', None))

## 5. `invoke` vs `stream` — resposta pronta vs pingando

- `invoke(msg)` bloqueia até a resposta **completa** e devolve um `AIMessage`. Use em grafos (o nó precisa do texto final).
- `stream(msg)` devolve um **gerador**: cada `for` entrega um pedacinho (`chunk`) com parte do `content`. Some todos os `content` e dá o texto final. Use em UI/chat (mostrar digitando).
- Erro clássico: tratar o retorno de `stream` como string pronta. Ele é iterável — precisa do `for` (ou `''.join(...)`).

In [ ]:
# CÉLULA 5 — stream + prova de que a soma fecha
print('pingos: ', end='')
acumulado = ''
for pedaco in llm.stream("Diga 'um dois três' e pare."):
    print(pedaco.content, end='', flush=True)
    acumulado += pedaco.content
print()
completo = llm.invoke("Diga 'um dois três' e pare.").content
print('stream somado == invoke?', acumulado == completo)

## 6. Variações que quebram (leia antes de rodar)

1. **Sem chave** → `get_llm` avisa e cai no Fake. Mecânica idêntica, texto simulado. É assim que os testes rodam no CI.
2. **`content` vazio (modelo de raciocínio)** → aconteceu de verdade com `openai/gpt-oss-20b` sem `reasoning_effort`: 2046 tokens gastos raciocinando, `finish_reason: length`, resposta nunca escrita. Diagnóstico e correção na seção 3.1.
3. **`temperature=0 vs 0.9`** → com Groq real, rode a CÉLULA 3 duas vezes trocando o parâmetro numa pergunta aberta (ex: 'invente um slogan') e compare: `0` repete, `0.9` varia. No Fake, as duas dão igual — o que importa é o código ser o mesmo.
4. **Modelo inexistente** (`model='modelo-que-nao-existe'`) → a API devolve 400/404. Em produção, capture com `try/except` e faça fallback (seu `AGENTS.md` prevê Groq → Ollama Cloud → OpenRouter).
5. **Kernel/venv errado** → `ModuleNotFoundError: langchain_groq`. Local: use o kernel do `.venv` (ou `uv run jupyter notebook`). No Colab: a célula 0 já instala.
6. **`reasoning_effort='none'`** → a API rejeita com 400 (`must be one of low, medium, high`). Não existe desligar o raciocínio, apenas dosá-lo.


In [ ]:
# CÉLULA 6 — desafio: exibir palavra por palavra
# Complete a função e rode: ela deve imprimir o texto com uma palavra por linha
# e DEVOLVER a lista de palavras. O assert valida (funciona no Fake e no Groq).
def exibir_palavras(llm, frase: str) -> list:
    texto = llm.invoke(frase).content
    palavras = texto.split()  # TODO: você pode tentar montar via stream também
    for p in palavras:
        print(p)
    return palavras

saida = exibir_palavras(llm, "Diga 'um dois três' e pare.")
assert isinstance(saida, list) and len(saida) >= 3, saida
print('OK —', len(saida), 'palavras.')


## 7. Checklist + próxima aula

**Você entendeu se consegue responder sem olhar:**
- [ ] Por que `uv sync` é mais reproduzível que `pip install -r requirements.txt`? E quando `pip` ainda é a escolha (Colab)?
- [ ] Onde mora a chave (`GROQ_API_KEY`) e onde ela NUNCA pode ir (git)?
- [ ] Qual a diferença entre raciocínio (`reasoning_content`) e resposta (`content`)? O que `reasoning_effort="low"` resolve e quando subir para `medium/high`?
- [ ] O que `temperature=0` garante na prática — e quando subir para `0.9`?
- [ ] Diferença de uso entre `invoke` (AIMessage pronto) e `stream` (gerador de chunks)?
- [ ] O que há dentro de um `AIMessage` além de `content`?

**Próxima aula (02 — Grafos):** refaremos a lanchonete da Aula 00 no padrão rico (dissecção + variações que quebram + erros de overwrite), reaproveitando `src/aula02_grafo.py` como base.